# Pre- und Post-Fire Woody Cover Trajektorien pro Brandjahr

Dieses Skript extrahiert für jedes Jahr mit Brandereignis (basierend auf einem Burned Area Multiband-Raster) die komplette Woody Cover Zeitreihe für alle Pixel, die im jeweiligen Jahr gebrannt haben.  
Für jedes Brandjahr wird ein separates Multiband-Raster erzeugt, das folgende Informationen enthält:

- **Alle Jahre vor dem Feuer**
- **Das Brandjahr selbst**
- **Alle Jahre nach dem Feuer**

Nur Pixel, die im jeweiligen Jahr gebrannt haben, enthalten Werte; alle anderen Pixel werden mit dem definierten `nodata`-Wert belegt.  
So erhält man für jedes Brandereignis eine vollständige Trajektorie des Woody Cover Verlaufs pro Pixel, die für weitere Analysen (z.B. Pre-/Post-Fire-Vergleiche) genutzt werden kann.

In [1]:
import rasterio
import numpy as np
import os
import time

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
burned_mask_path = os.path.join(
    workDir,
    r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_binary_yearly_500m.tif"
)
out_woody_path = os.path.join(
    workDir,
    r"_00_SHAPEFILES\woody_cover_500m_3035.tif"
)
out_dir = os.path.join(workDir, "_Runs", "02_Trajectories_Full_new")
os.makedirs(out_dir, exist_ok=True)

years_woody = list(range(1985, 2026))
years_burned = list(range(2000, 2026))
nodata_value = 255


print("Lade Woody Cover Raster ...")
start = time.time()
with rasterio.open(out_woody_path) as woody_src:
    woody = woody_src.read()  # (bands, y, x)
    meta = woody_src.meta.copy()
    meta.update(nodata=nodata_value, count=woody.shape[0])
print(f"Woody Cover geladen. Dauer: {time.time() - start:.2f} Sekunden")

print("Lade Burned Area Raster ...")
start = time.time()
with rasterio.open(burned_mask_path) as burned_src:
    burned = burned_src.read()  # (bands, y, x)
print(f"Burned Area geladen. Dauer: {time.time() - start:.2f} Sekunden")

for fire_idx, fire_year in enumerate(years_burned):
    print(f"\nBearbeite Brandjahr {fire_year} ...")
    step_start = time.time()
    fire_mask = burned[fire_idx] == 1

    print("Maskiere komplette Woody Cover Zeitreihe für gebrannte Pixel ...")
    woody_masked = np.where(fire_mask, woody, nodata_value)

    out_path = os.path.join(out_dir, f"woody_full_trajectory_burn_{fire_year}.tif")
    print(f"Speichere nach {out_path} ...")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(woody_masked)
    print(f"Gespeichert: {out_path} (Dauer: {time.time() - step_start:.2f} Sekunden)")

print("\nAlle Trajektorien wurden erfolgreich erstellt.")

Lade Woody Cover Raster ...
Woody Cover geladen. Dauer: 12.27 Sekunden
Lade Burned Area Raster ...
Burned Area geladen. Dauer: 7.82 Sekunden

Bearbeite Brandjahr 2000 ...
Maskiere komplette Woody Cover Zeitreihe für gebrannte Pixel ...
Speichere nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Trajectories_Full_new\woody_full_trajectory_burn_2000.tif ...
Gespeichert: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Trajectories_Full_new\woody_full_trajectory_burn_2000.tif (Dauer: 21.09 Sekunden)

Bearbeite Brandjahr 2001 ...
Maskiere komplette Woody Cover Zeitreihe für gebrannte Pixel ...
Speichere nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Trajectories_Full_new\woody_full_trajectory_burn_2001.tif ...
Gespeichert: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Trajectories_Full_new\woody_full_trajectory_burn_2001.tif (Dauer: 21.99 Sekunden)

Bearbeite Brandjahr 2002 ...
Maskiere komplette Woody Cover Zeitreihe für gebrannte Pixel ..

# als ein Multiband-Raster

In [2]:
import rasterio
import numpy as np
import os
import time

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
burned_mask_path = os.path.join(
    workDir,
    r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_binary_yearly_500m.tif"
)
out_woody_path = os.path.join(
    workDir,
    r"_00_SHAPEFILES\woody_cover_500m_3035.tif"
)
out_path = os.path.join(workDir, "_Runs", "02_Trajectories_Full_new", "woody_burned_combined_new.tif")

# Original year ranges - we'll adjust these based on actual data
years_woody_original = list(range(1985, 2026))   # 41 Jahre
years_burned = list(range(2000, 2026))  # 26 Jahre
nodata_value = 255

print("Lade Woody Cover Raster ...")
start = time.time()
with rasterio.open(out_woody_path) as woody_src:
    woody = woody_src.read()
    meta = woody_src.meta.copy()
print(f"Woody Cover geladen. Dauer: {time.time() - start:.2f} Sekunden")
print(f"Woody Cover Shape: {woody.shape}")

print("Lade Burned Area Raster ...")
start = time.time()
with rasterio.open(burned_mask_path) as burned_src:
    burned = burned_src.read()
print(f"Burned Area geladen. Dauer: {time.time() - start:.2f} Sekunden")
print(f"Burned Area Shape: {burned.shape}")

# Adjust years_woody to match actual data
actual_woody_bands = woody.shape[0]
actual_burned_bands = burned.shape[0]

# Create years list based on actual band count
# Assuming woody data starts from 1985 and has 40 bands (1985-2024)
years_woody = list(range(1985, 1985 + actual_woody_bands))
# Burned data should already match (2000-2025, 26 bands)

print(f"Adjusted years_woody to {actual_woody_bands} bands: {years_woody[0]}-{years_woody[-1]}")
print(f"Years_burned remains {actual_burned_bands} bands: {years_burned[0]}-{years_burned[-1]}")

# Kombiniere die Daten: erst woody, dann burned
combined = np.concatenate([woody, burned], axis=0)
meta.update(nodata=nodata_value, count=combined.shape[0])

print(f"Speichere kombinierten Raster mit {combined.shape[0]} Bändern nach {out_path} ...")
with rasterio.open(out_path, "w", **meta) as dst:
    dst.write(combined)
print("Kombinierter Raster gespeichert.")

# Setze Bandbeschreibungen
print("Setze Bandbeschreibungen ...")
with rasterio.open(out_path, "r+") as dst:
    # Woody cover bands (1 to actual_woody_bands)
    for i, year in enumerate(years_woody):
        dst.set_band_description(i+1, f"Woody Cover {year}")
    
    # Burned area bands (actual_woody_bands+1 to total)
    for i, year in enumerate(years_burned):
        band_index = len(years_woody) + i + 1
        dst.set_band_description(band_index, f"Burn Event {year}")

print("Bandbeschreibungen gesetzt.")

print("\nBandstruktur:")
for i, y in enumerate(years_woody):
    print(f"Band {i+1}: Woody Cover {y}")
for i, y in enumerate(years_burned):
    print(f"Band {len(years_woody)+i+1}: Burn Event {y} (1=Burn, 0=No Burn)")

# Debug: Print actual band count and expected ranges
print(f"\nDebug Info:")
print(f"Total bands: {combined.shape[0]}")
print(f"Years woody: {len(years_woody)} (bands 1-{len(years_woody)})")
print(f"Years burned: {len(years_burned)} (bands {len(years_woody)+1}-{len(years_woody)+len(years_burned)})")
print(f"Expected max band index: {len(years_woody)+len(years_burned)}")
print(f"Actual woody bands: {actual_woody_bands}")
print(f"Actual burned bands: {actual_burned_bands}")
print(f"Total actual bands: {actual_woody_bands + actual_burned_bands}")

Lade Woody Cover Raster ...
Woody Cover geladen. Dauer: 22.00 Sekunden
Woody Cover Shape: (40, 9660, 10483)
Lade Burned Area Raster ...
Burned Area geladen. Dauer: 12.37 Sekunden
Burned Area Shape: (26, 9660, 10483)
Adjusted years_woody to 40 bands: 1985-2024
Years_burned remains 26 bands: 2000-2025
Speichere kombinierten Raster mit 66 Bändern nach \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Trajectories_Full_new\woody_burned_combined_new.tif ...
Kombinierter Raster gespeichert.
Setze Bandbeschreibungen ...
Bandbeschreibungen gesetzt.

Bandstruktur:
Band 1: Woody Cover 1985
Band 2: Woody Cover 1986
Band 3: Woody Cover 1987
Band 4: Woody Cover 1988
Band 5: Woody Cover 1989
Band 6: Woody Cover 1990
Band 7: Woody Cover 1991
Band 8: Woody Cover 1992
Band 9: Woody Cover 1993
Band 10: Woody Cover 1994
Band 11: Woody Cover 1995
Band 12: Woody Cover 1996
Band 13: Woody Cover 1997
Band 14: Woody Cover 1998
Band 15: Woody Cover 1999
Band 16: Woody Cover 2000
Band 17: Woody Cover 20

In [4]:
# Eigenschaften des kombinierten Rasterstacks anzeigen
import rasterio
import numpy as np
import os
import time

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# Pfad zum kombinierten Rasterstack
out_path = os.path.join(workDir, "_Runs", "02_Trajectories_Full_new", "woody_burned_combined_new.tif")

print("=== Eigenschaften des Rasterstacks ===")
with rasterio.open(out_path) as src:
    print(f"Dateipfad: {out_path}")
    print(f"Anzahl Bänder: {src.count}")
    print(f"Breite (Pixel): {src.width}")
    print(f"Höhe (Pixel): {src.height}")
    print(f"Datentyp: {src.dtypes[0]}")
    print(f"NoData-Wert: {src.nodata}")
    print(f"Koordinatensystem (CRS): {src.crs}")
    print(f"Räumliche Auflösung: {src.res}")
    print(f"Bounding Box: {src.bounds}")
    print(f"Transform: {src.transform}")
    
    print(f"\n=== Bandbeschreibungen ===")
    descriptions = src.descriptions
    for i in range(min(10, src.count)):  # Erste 10 Bänder
        desc = descriptions[i] if descriptions[i] else f"Band {i+1}"
        print(f"Band {i+1}: {desc}")
    
    if src.count > 10:
        print("...")
        for i in range(max(0, src.count - 5), src.count):  # Letzte 5 Bänder
            desc = descriptions[i] if descriptions[i] else f"Band {i+1}"
            print(f"Band {i+1}: {desc}")

# Basierend auf den Bandbeschreibungen rekonstruieren wir die Jahresbereiche
print(f"\n=== Struktur des kombinierten Datensatzes ===")
# Aus den Bandbeschreibungen ableiten
woody_bands = []
burned_bands = []

with rasterio.open(out_path) as src:
    for i, desc in enumerate(src.descriptions):
        if desc and "Woody Cover" in desc:
            year = int(desc.split()[-1])
            woody_bands.append(year)
        elif desc and "Burn Event" in desc:
            year = int(desc.split()[-1])
            burned_bands.append(year)

if woody_bands and burned_bands:
    print(f"Woody Cover Jahre: {min(woody_bands)} - {max(woody_bands)} ({len(woody_bands)} Bänder)")
    print(f"Burned Area Jahre: {min(burned_bands)} - {max(burned_bands)} ({len(burned_bands)} Bänder)")
    print(f"Gesamtanzahl Bänder: {len(woody_bands) + len(burned_bands)}")
else:
    print("Konnte Jahresbereiche nicht aus Bandbeschreibungen extrahieren")
    print(f"Gesamtanzahl Bänder: {src.count}")

=== Eigenschaften des Rasterstacks ===
Dateipfad: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Trajectories_Full_new\woody_burned_combined_new.tif
Anzahl Bänder: 66
Breite (Pixel): 10483
Höhe (Pixel): 9660
Datentyp: uint8
NoData-Wert: 255.0
Koordinatensystem (CRS): EPSG:3035
Räumliche Auflösung: (500.0, 500.0)
Bounding Box: BoundingBox(left=2477500.0, bottom=1289000.0, right=7719000.0, top=6119000.0)
Transform: | 500.00, 0.00, 2477500.00|
| 0.00,-500.00, 6119000.00|
| 0.00, 0.00, 1.00|

=== Bandbeschreibungen ===
Band 1: Woody Cover 1985
Band 2: Woody Cover 1986
Band 3: Woody Cover 1987
Band 4: Woody Cover 1988
Band 5: Woody Cover 1989
Band 6: Woody Cover 1990
Band 7: Woody Cover 1991
Band 8: Woody Cover 1992
Band 9: Woody Cover 1993
Band 10: Woody Cover 1994
...
Band 62: Burn Event 2021
Band 63: Burn Event 2022
Band 64: Burn Event 2023
Band 65: Burn Event 2024
Band 66: Burn Event 2025

=== Struktur des kombinierten Datensatzes ===
Woody Cover Jahre: 1985 - 2024 (40 Bänder